In [1]:
using LinearAlgebra
using CSV
using DataFrames
using Statistics
using Random

In [ ]:

# Include solve_lle code
include("calculate_phase_split.jl")  # Contains solve_lle and read_tau_matrix functions


function morris_sensitivity_analysis(base_tau::Matrix{Float64}, 
                                     z_feed::Vector{Float64},
                                     delta::Float64=0.1,
                                     r::Int=4)
    
    println("="^50)
    println("Morris Sensitivity Analysis")
    println("="^50)
    println("Feed composition: $z_feed")
    println("Perturbation size: $(delta*100)%")
    println("Number of trajectories: $r")
    
    # Define the 6 off-diagonal tau parameters
    # For 3x3 matrix: τ_12, τ_13, τ_21, τ_23, τ_31, τ_32
    param_indices = [(1,2), (1,3), (2,1), (2,3), (3,1), (3,2)]
    param_names = ["τ₁₂", "τ₁₃", "τ₂₁", "τ₂₃", "τ₃₁", "τ₃₂"]
    n_params = length(param_indices)
    
    total_evals = (n_params + 1) * r
    println("Total LLE evaluations: $total_evals")
    println("Estimated time: $(round(total_evals * 3 / 60, digits=1)) hours")
    println("="^50)
    println()
    
    # Store elementary effects for each parameter
    elementary_effects = Dict(i => Float64[] for i in 1:n_params)
    
    eval_count = 0
    
    for traj in 1:r
        println("Trajectory $traj/$r")
        println("-"^40)
        
        # Starting point: base tau matrix
        current_tau = copy(base_tau)
        
        # Evaluate at starting point
        eval_count += 1
        println("  Eval $eval_count/$total_evals: Base case")
        x1_curr, x2_curr = solve_lle(z_feed, current_tau)
        curr_metric = norm(x1_curr - x2_curr)  # Phase separation metric
        
        # Random permutation of parameters for this trajectory
        param_order = randperm(n_params)
        
        # Step through each parameter
        for param_idx in param_order
            # Perturb parameter
            row, col = param_indices[param_idx]
            perturbed_tau = copy(current_tau)
            perturbed_tau[row, col] *= (1 + delta)
            
            # Eval
            eval_count += 1
            println("  Eval $eval_count/$total_evals: Perturb $(param_names[param_idx])")
            x1_pert, x2_pert = solve_lle(z_feed, perturbed_tau)
            pert_metric = norm(x1_pert - x2_pert)
            
            # Calculate elementary effect
            ee = (pert_metric - curr_metric) / delta
            push!(elementary_effects[param_idx], ee)
            
            println("    Elementary effect: $(round(ee, digits=4))")
            
            # Move to perturbed point for next step
            current_tau = perturbed_tau
            curr_metric = pert_metric
        end
        println()
    end
    
    # Analyze results
    println("="^50)
    println("Sensitivity Analysis Results")
    println("="^50)
    
    sensitivity_results = []
    
    for i in 1:n_params
        effects = elementary_effects[i]
        mu = mean(effects)
        mu_star = mean(abs.(effects))  # mean of absolute effects
        sigma = std(effects)
        
        push!(sensitivity_results, (
            parameter = param_names[i],
            mu = mu,
            mu_star = mu_star,
            sigma = sigma
        ))
        
        println("$(param_names[i]):")
        println("  μ* (importance): $(round(mu_star, digits=4))")
        println("  σ (interactions): $(round(sigma, digits=4))")
        println("  μ (directional): $(round(mu, digits=4))")
    end
    
    # Sort by importance (mu_star)
    sort!(sensitivity_results, by = x -> x.mu_star, rev=true)
    
    println()
    println("="^50)
    println("Ranking by Importance (μ*):")
    println("="^50)
    for (rank, result) in enumerate(sensitivity_results)
        println("$rank. $(result.parameter): μ* = $(round(result.mu_star, digits=4))")
    end
    println()
    
    # Save results
    results_df = DataFrame(sensitivity_results)
    CSV.write("morris_sensitivity_results.csv", results_df)
    println("Results saved to morris_sensitivity_results.csv")
    
    return results_df
end

function main()
    # Load fitted tau matrix
    tau_file = "fitted_tau_matrix_2_5.csv"
    base_tau = read_tau_matrix(tau_file)
    
    # Define feed composition within convex hull
    z_feed = [0.2, 0.01, 0.79]  # Water, Furfural, EA
    
    # Run Morris screening w/ 4 trajectories
    results = morris_sensitivity_analysis(base_tau, z_feed, 0.1, 4)
    
    println()
    println("Analysis complete")
    
    return results
end


main (generic function with 1 method)

In [3]:
results = main()

Loaded tau matrix from fitted_tau_matrix_2_5.csv:


3×3 Matrix{Float64}:
 0.0       3.00421    4.02985
 2.31201   0.0       -1.87704
 2.84445  -0.321599   0.0

Morris Sensitivity Analysis
Feed composition: [0.2, 0.01, 0.79]
Perturbation size: 10.0%
Number of trajectories: 4
Total LLE evaluations: 28
Estimated time: 1.4 hours

Trajectory 1/4
----------------------------------------
  Eval 1/28: Base case


┌ Warning: Initial position cannot be on the boundary of the box. Moving elements to the interior.
│ Element indices affected: [4, 5]
└ @ Optim ~/.julia/packages/Optim/7krni/src/multivariate/solvers/constrained/fminbox.jl:441


  Eval 2/28: Perturb τ₁₃
    Elementary effect: -0.0951


...

┌ Warning: Initial position cannot be on the boundary of the box. Moving elements to the interior.
│ Element indices affected: [4, 5]
└ @ Optim ~/.julia/packages/Optim/7krni/src/multivariate/solvers/constrained/fminbox.jl:441


  Eval 28/28: Perturb τ₁₂
    Elementary effect: 0.0


┌ Warning: Initial position cannot be on the boundary of the box. Moving elements to the interior.
│ Element indices affected: [4, 5]
└ @ Optim ~/.julia/packages/Optim/7krni/src/multivariate/solvers/constrained/fminbox.jl:441



Sensitivity Analysis Results
τ₁₂:
  μ* (importance): 0.0
  σ (interactions): 0.0
  μ (directional): 0.0
τ₁₃:
  μ* (importance): 0.1075
  σ (interactions): 0.0142
  μ (directional): -0.1075
τ₂₁:
  μ* (importance): 0.0
  σ (interactions): 0.0
  μ (directional): 0.0
τ₂₃:
  μ* (importance): 0.0
  σ (interactions): 0.0
  μ (directional): 0.0
τ₃₁:
  μ* (importance): 0.0868
  σ (interactions): 0.0143
  μ (directional): 0.0868
τ₃₂:
  μ* (importance): 0.0
  σ (interactions): 0.0
  μ (directional): -0.0

Ranking by Importance (μ*):
1. τ₁₃: μ* = 0.1075
2. τ₃₁: μ* = 0.0868
3. τ₃₂: μ* = 0.0
4. τ₁₂: μ* = 0.0
5. τ₂₁: μ* = 0.0
6. τ₂₃: μ* = 0.0

Results saved to morris_sensitivity_results.csv

Analysis complete


Row,parameter,mu,mu_star,sigma
,String,Float64,Float64,Float64
1,τ₁₃,-0.107457,0.107457,0.0142347
2,τ₃₁,0.0867534,0.0867534,0.0142525
3,τ₃₂,-2.02291e-5,3.07086e-5,3.58801e-5
4,τ₁₂,1.67266e-5,2.40124e-5,2.09059e-5
5,τ₂₁,9.74992e-6,2.14483e-5,3.27983e-5
6,τ₂₃,3.07376e-6,5.03422e-6,6.49352e-6
